## Install Required Libraries

In [ ]:
!pip install numpy scikit-learn pyts torch matplotlib sktime==0.30.0 --quiet
!pip install git+https://github.com/gon-uri/detach_rocket --quiet

## Download Dataset from UCR

In [ ]:
from detach_rocket.utils_datasets import fetch_ucr_dataset

# Download Dataset
dataset_name_list = ['FordB'] # PhalangesOutlinesCorrect ProximalPhalanxOutlineCorrect #Fordb
current_dataset = fetch_ucr_dataset(dataset_name_list[0])

In [ ]:
print(current_dataset.keys())

In [ ]:
X_train = current_dataset["data_train"]
y_train = current_dataset["target_train"]

X_test = current_dataset["data_test"]
y_test = current_dataset["target_test"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
import numpy as np

X_train_rocket = X_train[:, np.newaxis, :]
X_test_rocket = X_test[:, np.newaxis, :]

print("X_train_rocket:", X_train_rocket.shape)
print("X_test_rocket:", X_test_rocket.shape)

In [ ]:
from sktime.transformations.panel.rocket import Rocket

rocket_seeds = [0, 1, 2, 3, 4]
rocket_features_by_seed = {}

for seed in rocket_seeds:
    print(f"Running ROCKET with seed={seed}")

    rocket_tmp = Rocket(
        num_kernels=10_000,
        random_state=seed
    )

    rocket_tmp.fit(X_train_rocket)

    Z_train_tmp = rocket_tmp.transform(X_train_rocket)
    Z_test_tmp = rocket_tmp.transform(X_test_rocket)

    rocket_features_by_seed[seed] = {
        "train": Z_train_tmp,
        "test": Z_test_tmp
    }

    print(
        f"seed={seed}:",
        Z_train_tmp.shape,
        Z_test_tmp.shape
    )

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

clean_data_by_seed = {}

for seed in rocket_seeds:
    Z_train_seed = rocket_features_by_seed[seed]["train"]
    Z_test_seed = rocket_features_by_seed[seed]["test"]

    Z_sub_raw, Z_val_raw, y_sub, y_val = train_test_split(
        Z_train_seed,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=42
    )

    scaler = StandardScaler()

    Z_sub = scaler.fit_transform(Z_sub_raw)
    Z_val = scaler.transform(Z_val_raw)
    Z_test_scaled = scaler.transform(Z_test_seed)

    clean_data_by_seed[seed] = {
        "subtrain": Z_sub,
        "validation": Z_val,
        "test": Z_test_scaled,
        "y_sub": y_sub,
        "y_val": y_val
    }

    print(
        f"seed={seed}:",
        Z_sub.shape,
        Z_val.shape,
        Z_test_scaled.shape
    )

In [ ]:
import pandas as pd

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score

selectkbest_results = []

for seed in rocket_seeds:
    data = clean_data_by_seed[seed]

    Z_sub = data["subtrain"]
    Z_val = data["validation"]
    Z_test_scaled = data["test"]
    y_sub = data["y_sub"]

    selector = SelectKBest(
        score_func=f_classif,
        k=789
    )

    Z_sub_sel = selector.fit_transform(Z_sub, y_sub)
    Z_test_sel = selector.transform(Z_test_scaled)

    model = RidgeClassifierCV(
        alphas=np.logspace(-3, 3, 10)
    )

    model.fit(Z_sub_sel, y_sub)

    y_pred = model.predict(Z_test_sel)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    selectkbest_results.append({
        "seed": seed,
        "accuracy": acc,
        "f1": f1,
        "alpha": model.alpha_
    })

    print(
        f"seed={seed} | "
        f"Accuracy={acc:.4f} | "
        f"F1={f1:.4f} | "
        f"alpha={model.alpha_}"
    )

selectkbest_df = pd.DataFrame(selectkbest_results)

print()
print(selectkbest_df)

In [ ]:
print("SelectKBest multi-seed summary")
print(
    "Accuracy:",
    f"{selectkbest_df['accuracy'].mean()*100:.2f}% ± "
    f"{selectkbest_df['accuracy'].std(ddof=1)*100:.2f}%"
)
print(
    "F1-score:",
    f"{selectkbest_df['f1'].mean()*100:.2f}% ± "
    f"{selectkbest_df['f1'].std(ddof=1)*100:.2f}%"
)

In [ ]:
print(type(current_dataset))
print(current_dataset.keys())

In [ ]:
print("X_train shape:", current_dataset.data_train.shape)
print("y_train shape:", current_dataset.target_train.shape)

print("X_test shape:", current_dataset.data_test.shape)
print("y_test shape:", current_dataset.target_test.shape)

In [ ]:
import numpy as np

print("Train classes:", np.unique(
    current_dataset.target_train,
    return_counts=True
))

print("Test classes:", np.unique(
    current_dataset.target_test,
    return_counts=True
))

In [ ]:
X_train = current_dataset.data_train
y_train = current_dataset.target_train

X_test = current_dataset.data_test
y_test = current_dataset.target_test

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
print("Missing values in X_train:", np.isnan(X_train).sum())
print("Missing values in X_test:", np.isnan(X_test).sum())

In [ ]:
import matplotlib.pyplot as plt

idx_neg = np.where(y_train == -1)[0][0]
idx_pos = np.where(y_train == 1)[0][0]

plt.figure(figsize=(10, 4))
plt.plot(X_train[idx_neg], label="Class -1")
plt.plot(X_train[idx_pos], label="Class +1")
plt.xlabel("Time point")
plt.ylabel("Signal value")
plt.title("Example time series from FordB")
plt.legend()
plt.show()

In [ ]:
X_train_rocket = X_train[:, np.newaxis, :]
X_test_rocket = X_test[:, np.newaxis, :]

print("X_train_rocket shape:", X_train_rocket.shape)
print("X_test_rocket shape:", X_test_rocket.shape)

In [ ]:
from sktime.transformations.panel.rocket import Rocket

rocket = Rocket(
    num_kernels=10_000,
    random_state=42
)

rocket.fit(X_train_rocket)

Z_train = rocket.transform(X_train_rocket)
Z_test = rocket.transform(X_test_rocket)

print("Z_train shape:", Z_train.shape)
print("Z_test shape:", Z_test.shape)

In [ ]:
rocket_seeds = [0, 1, 2, 3, 4]

print("ROCKET seeds:", rocket_seeds)

In [ ]:
import sktime
print(sktime.__version__)

In [ ]:
from sktime.transformations.panel.rocket import Rocket

rocket_seeds = [0, 1, 2, 3, 4]
rocket_features_by_seed = {}

for seed in rocket_seeds:
    print(f"Running ROCKET with seed={seed}")

    rocket_tmp = Rocket(
        num_kernels=10_000,
        random_state=seed
    )

    rocket_tmp.fit(X_train_rocket)

    Z_train_tmp = rocket_tmp.transform(X_train_rocket)
    Z_test_tmp = rocket_tmp.transform(X_test_rocket)

    rocket_features_by_seed[seed] = {
        "train": Z_train_tmp,
        "test": Z_test_tmp
    }

    print(seed, Z_train_tmp.shape, Z_test_tmp.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score
from time import perf_counter

scaler = StandardScaler()

Z_train_scaled = scaler.fit_transform(Z_train)
Z_test_scaled = scaler.transform(Z_test)

alphas = np.logspace(-3, 3, 10)

full_model = RidgeClassifierCV(alphas=alphas)

start = perf_counter()
full_model.fit(Z_train_scaled, y_train)
training_time = perf_counter() - start

start = perf_counter()
y_pred = full_model.predict(Z_test_scaled)
prediction_time = perf_counter() - start

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("Training time:", training_time)
print("Prediction time:", prediction_time)
print("Selected alpha:", full_model.alpha_)

In [ ]:
baseline_results = {
    "model": "Full ROCKET",
    "num_features": Z_train_scaled.shape[1],
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "training_time": training_time,
    "prediction_time": prediction_time,
    "alpha": full_model.alpha_
}

baseline_results

In [ ]:
from sklearn.model_selection import train_test_split

Z_train_sub, Z_val, y_train_sub, y_val = train_test_split(
    Z_train_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Training subset:", Z_train_sub.shape, y_train_sub.shape)
print("Validation subset:", Z_val.shape, y_val.shape)
print("Test set:", Z_test_scaled.shape, y_test.shape)

In [ ]:
from detach_rocket.detach_classes import DetachMatrix
import inspect

print(inspect.signature(DetachMatrix))

In [ ]:
print(inspect.signature(DetachMatrix.fit))

In [ ]:
detach_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

print(detach_model)

In [ ]:
start = perf_counter()

detach_model.fit(
    Z_train_sub,
    y_train_sub,
    val_set=Z_val,
    val_set_y=y_val
)

detach_time = perf_counter() - start

print("Detach fitting time:", detach_time)

In [ ]:
print([name for name in dir(detach_model) if not name.startswith("_")])

In [ ]:
start = perf_counter()

detach_pred = detach_model.predict(Z_test_scaled)

detach_prediction_time = perf_counter() - start

detach_accuracy = accuracy_score(y_test, detach_pred)
detach_f1 = f1_score(y_test, detach_pred)

print("DETACH Accuracy:", detach_accuracy)
print("DETACH F1-score:", detach_f1)
print("DETACH prediction time:", detach_prediction_time)

In [ ]:
print(detach_model.__dict__.keys())

In [ ]:
selected_features = int(np.sum(detach_model._feature_mask))
total_features = len(detach_model._feature_mask)
retained_percentage = 100 * selected_features / total_features
reduction_percentage = 100 - retained_percentage

print("Selected features:", selected_features)
print("Total features:", total_features)
print("Retained percentage:", retained_percentage)
print("Feature reduction:", reduction_percentage)

In [ ]:
detach_results = {
    "model": "DETACH-ROCKET",
    "num_features": selected_features,
    "retained_percentage": retained_percentage,
    "feature_reduction": reduction_percentage,
    "accuracy": detach_accuracy,
    "f1_score": detach_f1,
    "selection_time": detach_time,
    "prediction_time": detach_prediction_time
}

detach_results

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame([
    {
        "Model": "Full ROCKET",
        "Features": baseline_results["num_features"],
        "Accuracy": baseline_results["accuracy"],
        "F1-score": baseline_results["f1_score"],
        "Prediction Time": baseline_results["prediction_time"]
    },
    {
        "Model": "DETACH-ROCKET",
        "Features": detach_results["num_features"],
        "Accuracy": detach_results["accuracy"],
        "F1-score": detach_results["f1_score"],
        "Prediction Time": detach_results["prediction_time"]
    }
])

comparison_df

In [ ]:
comparison_df.to_csv("comparison_results.csv", index=False)
print("Saved successfully")

In [ ]:
comparison_df.plot(
    x="Model",
    y=["Accuracy", "F1-score"],
    kind="bar",
    figsize=(8, 4)
)

plt.ylim(0.75, 0.85)
plt.ylabel("Score")
plt.title("Full ROCKET vs DETACH-ROCKET")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
comparison_df.plot(
    x="Model",
    y="Features",
    kind="bar",
    figsize=(7, 4),
    legend=False
)

plt.ylabel("Number of Features")
plt.title("Feature Count Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(
    comparison_df["Model"],
    comparison_df["Features"]
)

plt.ylabel("Number of Features")
plt.title("Feature Count Comparison")
plt.tight_layout()

plt.savefig(
    "feature_count_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
rng = np.random.default_rng(42)

random_feature_indices = rng.choice(
    Z_train_scaled.shape[1],
    size=selected_features,
    replace=False
)

print("Randomly selected features:", len(random_feature_indices))

In [ ]:
random_model = RidgeClassifierCV(alphas=alphas)

start = perf_counter()

random_model.fit(
    Z_train_scaled[:, random_feature_indices],
    y_train
)

random_training_time = perf_counter() - start

start = perf_counter()

random_pred = random_model.predict(
    Z_test_scaled[:, random_feature_indices]
)

random_prediction_time = perf_counter() - start

random_accuracy = accuracy_score(y_test, random_pred)
random_f1 = f1_score(y_test, random_pred)

print("Random Pruning Accuracy:", random_accuracy)
print("Random Pruning F1-score:", random_f1)
print("Random Pruning Training Time:", random_training_time)
print("Random Pruning Prediction Time:", random_prediction_time)
print("Selected Alpha:", random_model.alpha_)

In [ ]:
random_results = {
    "model": "Random Pruning",
    "num_features": selected_features,
    "accuracy": random_accuracy,
    "f1_score": random_f1,
    "training_time": random_training_time,
    "prediction_time": random_prediction_time,
    "alpha": random_model.alpha_
}

random_results

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Full ROCKET",
        "Features": baseline_results["num_features"],
        "Accuracy": baseline_results["accuracy"],
        "F1-score": baseline_results["f1_score"],
        "Prediction Time": baseline_results["prediction_time"]
    },
    {
        "Model": "DETACH-ROCKET",
        "Features": detach_results["num_features"],
        "Accuracy": detach_results["accuracy"],
        "F1-score": detach_results["f1_score"],
        "Prediction Time": detach_results["prediction_time"]
    },
    {
        "Model": "Random Pruning",
        "Features": random_results["num_features"],
        "Accuracy": random_results["accuracy"],
        "F1-score": random_results["f1_score"],
        "Prediction Time": random_results["prediction_time"]
    }
])

comparison_df

In [ ]:
comparison_df.to_csv(
    "comparison_results_3_models.csv",
    index=False
)

print("Saved successfully")

In [ ]:
random_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

print("Number of random runs:", len(random_seeds))

In [ ]:
random_runs = []

for seed in random_seeds:
    rng = np.random.default_rng(seed)

    indices = rng.choice(
        Z_train_scaled.shape[1],
        size=selected_features,
        replace=False
    )

    model = RidgeClassifierCV(alphas=alphas)

    model.fit(
        Z_train_scaled[:, indices],
        y_train
    )

    predictions = model.predict(
        Z_test_scaled[:, indices]
    )

    random_runs.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, predictions),
        "f1_score": f1_score(y_test, predictions)
    })

print("Completed runs:", len(random_runs))

In [ ]:
random_runs_df = pd.DataFrame(random_runs)

print(random_runs_df)

print("\nMean Accuracy:", random_runs_df["accuracy"].mean())
print("Std Accuracy:", random_runs_df["accuracy"].std())

print("\nMean F1-score:", random_runs_df["f1_score"].mean())
print("Std F1-score:", random_runs_df["f1_score"].std())

In [ ]:
random_runs_df.to_csv(
    "random_pruning_10_runs.csv",
    index=False
)

print("Random runs saved successfully")

In [ ]:
detach_seeds = [0, 1, 2, 3, 4]

print("Number of DETACH runs:", len(detach_seeds))

In [ ]:
detach_runs = []

print("Result container is ready.")

In [ ]:
seed = detach_seeds[0]

print("Running seed:", seed)

# 1) ساخت ROCKET جدید با seed متفاوت
rocket_seed = Rocket(
    num_kernels=10_000,
    random_state=seed
)

rocket_seed.fit(X_train_rocket)

# 2) تولید ۲۰هزار ویژگی جدید
Z_train_seed = rocket_seed.transform(X_train_rocket)
Z_test_seed = rocket_seed.transform(X_test_rocket)

# 3) استانداردسازی ویژگی‌ها
scaler_seed = StandardScaler()

Z_train_seed_scaled = scaler_seed.fit_transform(Z_train_seed)
Z_test_seed_scaled = scaler_seed.transform(Z_test_seed)

# 4) جداسازی Train و Validation
Z_sub_seed, Z_val_seed, y_sub_seed, y_val_seed = train_test_split(
    Z_train_seed_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Prepared shapes:")
print(Z_sub_seed.shape)
print(Z_val_seed.shape)
print(Z_test_seed_scaled.shape)

In [ ]:
detach_seed_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

detach_seed_model.fit(
    Z_sub_seed,
    y_sub_seed,
    Z_val_seed,
    y_val_seed
)

print("DETACH seed 0 fitted successfully.")

In [ ]:
detach_seed_pred = detach_seed_model.predict(
    Z_test_seed_scaled
)

detach_seed_accuracy = accuracy_score(
    y_test,
    detach_seed_pred
)

detach_seed_f1 = f1_score(
    y_test,
    detach_seed_pred
)

print("DETACH seed 0 Accuracy:", detach_seed_accuracy)
print("DETACH seed 0 F1-score:", detach_seed_f1)

In [ ]:
detach_runs.append({
    "seed": seed,
    "accuracy": detach_seed_accuracy,
    "f1_score": detach_seed_f1
})

print(detach_runs)

In [ ]:
seed = detach_seeds[1]

print("Next DETACH seed:", seed)

In [ ]:
rocket_seed = Rocket(
    num_kernels=10_000,
    random_state=seed
)

rocket_seed.fit(X_train_rocket)

Z_train_seed = rocket_seed.transform(X_train_rocket)
Z_test_seed = rocket_seed.transform(X_test_rocket)

scaler_seed = StandardScaler()

Z_train_seed_scaled = scaler_seed.fit_transform(Z_train_seed)
Z_test_seed_scaled = scaler_seed.transform(Z_test_seed)

Z_sub_seed, Z_val_seed, y_sub_seed, y_val_seed = train_test_split(
    Z_train_seed_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Prepared seed:", seed)
print(Z_sub_seed.shape)
print(Z_val_seed.shape)
print(Z_test_seed_scaled.shape)

In [ ]:
detach_seed_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

detach_seed_model.fit(
    Z_sub_seed,
    y_sub_seed,
    Z_val_seed,
    y_val_seed
)

print("DETACH seed 1 fitted successfully.")

In [ ]:
detach_seed_pred = detach_seed_model.predict(
    Z_test_seed_scaled
)

detach_seed_accuracy = accuracy_score(
    y_test,
    detach_seed_pred
)

detach_seed_f1 = f1_score(
    y_test,
    detach_seed_pred
)

print("DETACH seed 1 Accuracy:", detach_seed_accuracy)
print("DETACH seed 1 F1-score:", detach_seed_f1)

In [ ]:
detach_runs.append({
    "seed": seed,
    "accuracy": detach_seed_accuracy,
    "f1_score": detach_seed_f1
})

print(detach_runs)

In [ ]:
seed = detach_seeds[2]

print("Next DETACH seed:", seed)

In [ ]:
rocket_seed = Rocket(
    num_kernels=10_000,
    random_state=seed
)

rocket_seed.fit(X_train_rocket)

Z_train_seed = rocket_seed.transform(X_train_rocket)
Z_test_seed = rocket_seed.transform(X_test_rocket)

scaler_seed = StandardScaler()

Z_train_seed_scaled = scaler_seed.fit_transform(Z_train_seed)
Z_test_seed_scaled = scaler_seed.transform(Z_test_seed)

Z_sub_seed, Z_val_seed, y_sub_seed, y_val_seed = train_test_split(
    Z_train_seed_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Prepared seed:", seed)
print(Z_sub_seed.shape)
print(Z_val_seed.shape)
print(Z_test_seed_scaled.shape)

In [ ]:
detach_seed_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

detach_seed_model.fit(
    Z_sub_seed,
    y_sub_seed,
    Z_val_seed,
    y_val_seed
)

print("DETACH seed 2 fitted successfully.")

In [ ]:
detach_seed_pred = detach_seed_model.predict(
    Z_test_seed_scaled
)

detach_seed_accuracy = accuracy_score(
    y_test,
    detach_seed_pred
)

detach_seed_f1 = f1_score(
    y_test,
    detach_seed_pred
)

print("DETACH seed 2 Accuracy:", detach_seed_accuracy)
print("DETACH seed 2 F1-score:", detach_seed_f1)

In [ ]:
detach_runs.append({
    "seed": seed,
    "accuracy": detach_seed_accuracy,
    "f1_score": detach_seed_f1
})

print(detach_runs)

In [ ]:
seed = detach_seeds[3]

print("Next DETACH seed:", seed)

In [ ]:
rocket_seed = Rocket(
    num_kernels=10_000,
    random_state=seed
)

rocket_seed.fit(X_train_rocket)

Z_train_seed = rocket_seed.transform(X_train_rocket)
Z_test_seed = rocket_seed.transform(X_test_rocket)

scaler_seed = StandardScaler()

Z_train_seed_scaled = scaler_seed.fit_transform(Z_train_seed)
Z_test_seed_scaled = scaler_seed.transform(Z_test_seed)

Z_sub_seed, Z_val_seed, y_sub_seed, y_val_seed = train_test_split(
    Z_train_seed_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Prepared seed:", seed)
print(Z_sub_seed.shape)
print(Z_val_seed.shape)
print(Z_test_seed_scaled.shape)

In [ ]:
detach_seed_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

detach_seed_model.fit(
    Z_sub_seed,
    y_sub_seed,
    Z_val_seed,
    y_val_seed
)

print("DETACH seed 3 fitted successfully.")

In [ ]:
detach_seed_pred = detach_seed_model.predict(
    Z_test_seed_scaled
)

detach_seed_accuracy = accuracy_score(
    y_test,
    detach_seed_pred
)

detach_seed_f1 = f1_score(
    y_test,
    detach_seed_pred
)

print("DETACH seed 3 Accuracy:", detach_seed_accuracy)
print("DETACH seed 3 F1-score:", detach_seed_f1)

In [ ]:
detach_runs.append({
    "seed": seed,
    "accuracy": detach_seed_accuracy,
    "f1_score": detach_seed_f1
})

print(detach_runs)

In [ ]:
seed = detach_seeds[4]

print("Next DETACH seed:", seed)

In [ ]:
rocket_seed = Rocket(
    num_kernels=10_000,
    random_state=seed
)

rocket_seed.fit(X_train_rocket)

Z_train_seed = rocket_seed.transform(X_train_rocket)
Z_test_seed = rocket_seed.transform(X_test_rocket)

scaler_seed = StandardScaler()

Z_train_seed_scaled = scaler_seed.fit_transform(Z_train_seed)
Z_test_seed_scaled = scaler_seed.transform(Z_test_seed)

Z_sub_seed, Z_val_seed, y_sub_seed, y_val_seed = train_test_split(
    Z_train_seed_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Prepared seed:", seed)
print(Z_sub_seed.shape)
print(Z_val_seed.shape)
print(Z_test_seed_scaled.shape)

In [ ]:
detach_seed_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

detach_seed_model.fit(
    Z_sub_seed,
    y_sub_seed,
    Z_val_seed,
    y_val_seed
)

print("DETACH seed 4 fitted successfully.")

In [ ]:
detach_seed_pred = detach_seed_model.predict(
    Z_test_seed_scaled
)

detach_seed_accuracy = accuracy_score(
    y_test,
    detach_seed_pred
)

detach_seed_f1 = f1_score(
    y_test,
    detach_seed_pred
)

print("DETACH seed 4 Accuracy:", detach_seed_accuracy)
print("DETACH seed 4 F1-score:", detach_seed_f1)

In [ ]:
detach_runs.append({
    "seed": seed,
    "accuracy": detach_seed_accuracy,
    "f1_score": detach_seed_f1
})

print(detach_runs)

In [ ]:
detach_runs_df = pd.DataFrame(detach_runs)

print(detach_runs_df)

print("\nMean Accuracy:", detach_runs_df["accuracy"].mean())
print("Std Accuracy:", detach_runs_df["accuracy"].std())

print("\nMean F1-score:", detach_runs_df["f1_score"].mean())
print("Std F1-score:", detach_runs_df["f1_score"].std())

In [ ]:
detach_runs_df.to_csv(
    "detach_5_runs.csv",
    index=False
)

print("DETACH runs saved successfully")

In [ ]:
summary_results = pd.DataFrame([
    {
        "Method": "DETACH-ROCKET",
        "Runs": len(detach_runs_df),
        "Mean Accuracy": detach_runs_df["accuracy"].mean(),
        "Std Accuracy": detach_runs_df["accuracy"].std(),
        "Mean F1-score": detach_runs_df["f1_score"].mean(),
        "Std F1-score": detach_runs_df["f1_score"].std()
    },
    {
        "Method": "Random Pruning",
        "Runs": len(random_runs_df),
        "Mean Accuracy": random_runs_df["accuracy"].mean(),
        "Std Accuracy": random_runs_df["accuracy"].std(),
        "Mean F1-score": random_runs_df["f1_score"].mean(),
        "Std F1-score": random_runs_df["f1_score"].std()
    }
])

summary_results

In [ ]:
summary_results.to_csv(
    "summary_results.csv",
    index=False
)

print("Summary results saved successfully")

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    summary_results["Method"],
    summary_results["Mean Accuracy"],
    yerr=summary_results["Std Accuracy"],
    capsize=6
)

plt.ylabel("Accuracy")
plt.title("DETACH-ROCKET vs Random Pruning")
plt.ylim(0.75, 0.83)
plt.tight_layout()

plt.savefig(
    "accuracy_mean_std_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    summary_results["Method"],
    summary_results["Mean F1-score"],
    yerr=summary_results["Std F1-score"],
    capsize=6
)

plt.ylabel("F1-score")
plt.title("DETACH-ROCKET vs Random Pruning")
plt.ylim(0.75, 0.83)
plt.tight_layout()

plt.savefig(
    "f1_mean_std_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
report_table = summary_results.copy()

for column in [
    "Mean Accuracy",
    "Std Accuracy",
    "Mean F1-score",
    "Std F1-score"
]:
    report_table[column] = report_table[column] * 100

report_table = report_table.round(2)

report_table

In [ ]:
report_table.to_csv(
    "report_table_percent.csv",
    index=False
)

print("Report table saved successfully")

In [ ]:
accuracy_improvement = (
    detach_runs_df["accuracy"].mean()
    - random_runs_df["accuracy"].mean()
) * 100

f1_improvement = (
    detach_runs_df["f1_score"].mean()
    - random_runs_df["f1_score"].mean()
) * 100

print("Accuracy improvement:", round(accuracy_improvement, 2), "percentage points")
print("F1-score improvement:", round(f1_improvement, 2), "percentage points")

In [ ]:
improvement_results = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "Improvement_percentage_points": accuracy_improvement
    },
    {
        "Metric": "F1-score",
        "Improvement_percentage_points": f1_improvement
    }
])

improvement_results.to_csv(
    "detach_vs_random_improvement.csv",
    index=False
)

print("Improvement results saved successfully")

In [ ]:
raw_scaler = StandardScaler()

X_train_raw_scaled = raw_scaler.fit_transform(X_train)
X_test_raw_scaled = raw_scaler.transform(X_test)

print("Raw train shape:", X_train_raw_scaled.shape)
print("Raw test shape:", X_test_raw_scaled.shape)

In [ ]:
raw_ridge_model = RidgeClassifierCV(alphas=alphas)

start = perf_counter()

raw_ridge_model.fit(
    X_train_raw_scaled,
    y_train
)

raw_ridge_training_time = perf_counter() - start

start = perf_counter()

raw_ridge_pred = raw_ridge_model.predict(
    X_test_raw_scaled
)

raw_ridge_prediction_time = perf_counter() - start

raw_ridge_accuracy = accuracy_score(
    y_test,
    raw_ridge_pred
)

raw_ridge_f1 = f1_score(
    y_test,
    raw_ridge_pred
)

print("Raw Ridge Accuracy:", raw_ridge_accuracy)
print("Raw Ridge F1-score:", raw_ridge_f1)
print("Raw Ridge Training Time:", raw_ridge_training_time)
print("Raw Ridge Prediction Time:", raw_ridge_prediction_time)
print("Selected Alpha:", raw_ridge_model.alpha_)

In [ ]:
raw_ridge_results = {
    "model": "Raw Ridge",
    "num_features": X_train_raw_scaled.shape[1],
    "accuracy": raw_ridge_accuracy,
    "f1_score": raw_ridge_f1,
    "training_time": raw_ridge_training_time,
    "prediction_time": raw_ridge_prediction_time,
    "alpha": raw_ridge_model.alpha_
}

raw_ridge_results

In [ ]:
comparison_df = pd.concat([
    comparison_df,
    pd.DataFrame([{
        "Model": "Raw Ridge",
        "Features": raw_ridge_results["num_features"],
        "Accuracy": raw_ridge_results["accuracy"],
        "F1-score": raw_ridge_results["f1_score"],
        "Prediction Time": raw_ridge_results["prediction_time"]
    }])
], ignore_index=True)

comparison_df

In [ ]:
comparison_df.to_csv(
    "comparison_results_4_models.csv",
    index=False
)

print("Four-model comparison saved successfully")

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    comparison_df["Model"],
    comparison_df["Accuracy"]
)

plt.ylabel("Accuracy")
plt.title("Accuracy Comparison of Four Models")
plt.ylim(0.4, 0.85)
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    "four_model_accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    comparison_df["Model"],
    comparison_df["Features"]
)

plt.ylabel("Number of Features")
plt.title("Feature Count Comparison")
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    "four_model_feature_count_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
feature_reduction_percent = (
    1 - (
        detach_results["num_features"]
        / baseline_results["num_features"]
    )
) * 100

print(
    "Feature reduction:",
    round(feature_reduction_percent, 2),
    "%"
)

In [ ]:
feature_reduction_results = pd.DataFrame([{
    "Full_ROCKET_Features": baseline_results["num_features"],
    "DETACH_Features": detach_results["num_features"],
    "Feature_Reduction_Percent": feature_reduction_percent
}])

feature_reduction_results.to_csv(
    "feature_reduction_results.csv",
    index=False
)

print("Feature reduction results saved successfully")

In [ ]:
final_report_table = comparison_df.copy()

final_report_table["Accuracy (%)"] = (
    final_report_table["Accuracy"] * 100
).round(2)

final_report_table["F1-score (%)"] = (
    final_report_table["F1-score"] * 100
).round(2)

final_report_table["Prediction Time (ms)"] = (
    final_report_table["Prediction Time"] * 1000
).round(3)

final_report_table = final_report_table[
    [
        "Model",
        "Features",
        "Accuracy (%)",
        "F1-score (%)",
        "Prediction Time (ms)"
    ]
]

final_report_table

In [ ]:
final_report_table.to_csv(
    "final_report_table.csv",
    index=False
)

print("Final report table saved successfully")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
detach_cm = confusion_matrix(
    y_test,
    detach_seed_pred,
    labels=[-1, 1]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=detach_cm,
    display_labels=[-1, 1]
)

display.plot()
plt.title("DETACH-ROCKET Confusion Matrix")
plt.tight_layout()

plt.savefig(
    "detach_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print(detach_cm)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        detach_seed_pred,
        labels=[-1, 1],
        digits=4
    )
)

In [ ]:
report_text = classification_report(
    y_test,
    detach_seed_pred,
    labels=[-1, 1],
    digits=4
)

with open(
    "detach_classification_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report_text)

print("Classification report saved successfully")

In [ ]:
raw_ridge_cm = confusion_matrix(
    y_test,
    raw_ridge_pred,
    labels=[-1, 1]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=raw_ridge_cm,
    display_labels=[-1, 1]
)

display.plot()
plt.title("Raw Ridge Confusion Matrix")
plt.tight_layout()

plt.savefig(
    "raw_ridge_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print(raw_ridge_cm)

In [ ]:
print(
    classification_report(
        y_test,
        raw_ridge_pred,
        labels=[-1, 1],
        digits=4
    )
)

In [ ]:
raw_report_text = classification_report(
    y_test,
    raw_ridge_pred,
    labels=[-1, 1],
    digits=4
)

with open(
    "raw_ridge_classification_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(raw_report_text)

print("Raw Ridge classification report saved successfully")

In [ ]:
classwise_summary = pd.DataFrame([
    {
        "Model": "DETACH-ROCKET",
        "Class": "-1",
        "Precision": 0.8120,
        "Recall": 0.7756,
        "F1-score": 0.7934
    },
    {
        "Model": "DETACH-ROCKET",
        "Class": "1",
        "Precision": 0.7892,
        "Recall": 0.8240,
        "F1-score": 0.8062
    },
    {
        "Model": "Raw Ridge",
        "Class": "-1",
        "Precision": 0.4851,
        "Recall": 0.5287,
        "F1-score": 0.5060
    },
    {
        "Model": "Raw Ridge",
        "Class": "1",
        "Precision": 0.4933,
        "Recall": 0.4499,
        "F1-score": 0.4706
    }
])

classwise_summary

In [ ]:
classwise_summary.to_csv(
    "classwise_summary.csv",
    index=False
)

print("Classwise summary saved successfully")

In [ ]:
pivot_f1 = classwise_summary.pivot(
    index="Class",
    columns="Model",
    values="F1-score"
)

pivot_f1.plot(
    kind="bar",
    figsize=(7, 5)
)

plt.ylabel("F1-score")
plt.title("Class-wise F1-score Comparison")
plt.xticks(rotation=0)
plt.ylim(0.4, 0.85)
plt.tight_layout()

plt.savefig(
    "classwise_f1_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
pivot_f1.to_csv(
    "classwise_f1_pivot.csv"
)

print("Class-wise F1 pivot saved successfully")

In [ ]:
project_summary = f"""
FordB Classification Summary

Full ROCKET:
- Features: {baseline_results['num_features']}
- Accuracy: {baseline_results['accuracy'] * 100:.2f}%
- F1-score: {baseline_results['f1_score'] * 100:.2f}%

DETACH-ROCKET:
- Selected features: {detach_results['num_features']}
- Feature reduction: {feature_reduction_percent:.2f}%
- Mean Accuracy over 5 runs: {detach_runs_df['accuracy'].mean() * 100:.2f}% ± {detach_runs_df['accuracy'].std() * 100:.2f}%
- Mean F1-score over 5 runs: {detach_runs_df['f1_score'].mean() * 100:.2f}% ± {detach_runs_df['f1_score'].std() * 100:.2f}%

Random Pruning:
- Selected features: {selected_features}
- Mean Accuracy over 10 runs: {random_runs_df['accuracy'].mean() * 100:.2f}% ± {random_runs_df['accuracy'].std() * 100:.2f}%
- Mean F1-score over 10 runs: {random_runs_df['f1_score'].mean() * 100:.2f}% ± {random_runs_df['f1_score'].std() * 100:.2f}%

Raw Ridge:
- Features: {raw_ridge_results['num_features']}
- Accuracy: {raw_ridge_results['accuracy'] * 100:.2f}%
- F1-score: {raw_ridge_results['f1_score'] * 100:.2f}%

DETACH improvement over Random Pruning:
- Accuracy: {accuracy_improvement:.2f} percentage points
- F1-score: {f1_improvement:.2f} percentage points
"""

with open(
    "project_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(project_summary)

print(project_summary)
print("Project summary saved successfully")

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import sktime
import matplotlib

environment_info = f"""
Python: {sys.version}
NumPy: {np.__version__}
Pandas: {pd.__version__}
Scikit-learn: {sklearn.__version__}
sktime: {sktime.__version__}
Matplotlib: {matplotlib.__version__}
"""

with open(
    "environment_versions.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(environment_info)

print(environment_info)
print("Environment information saved successfully")

In [ ]:
requirements_text = """
numpy
pandas
scikit-learn
matplotlib
sktime==0.30.0
torch
pyts
"""

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(requirements_text.strip())

print("requirements.txt saved successfully")

In [ ]:
from textwrap import dedent

readme_text = dedent("""
# DETACH-ROCKET on the FordB Dataset

## Project Overview

This project evaluates DETACH-ROCKET for binary time-series classification using the FordB dataset.

The study compares four approaches:

1. Full ROCKET with all 20,000 transformed features
2. DETACH-ROCKET feature selection
3. Random feature pruning using the same number of features selected by DETACH
4. RidgeClassifierCV applied directly to the raw time-series samples

## Dataset

FordB is a binary time-series classification dataset.

- Training samples: 3,636
- Test samples: 810
- Time points per sample: 500
- Classes: -1 and 1

## Main Results

- Full ROCKET features: 20,000
- DETACH-selected features: 789
- Feature reduction: 96.06%
- DETACH mean Accuracy over 5 runs: 80.69% ± 0.69%
- DETACH mean F1-score over 5 runs: 80.87% ± 0.51%
- Random pruning mean Accuracy over 10 runs: 79.07% ± 0.90%
- Random pruning mean F1-score over 10 runs: 79.03% ± 0.86%
- Raw Ridge Accuracy: 48.89%
- Raw Ridge F1-score: 47.06%

DETACH outperformed random pruning while retaining the same number of features. This indicates that its performance is related to informed feature selection rather than feature-count reduction alone.

## Installation

Create and activate a Python virtual environment, then install the required packages:

    pip install -r requirements.txt

## Experimental Pipeline

1. Load the FordB training and test sets.
2. Generate 20,000 ROCKET features.
3. Standardize the transformed features.
4. Train the Full ROCKET Ridge classifier.
5. Apply DETACH using a training-validation split.
6. Evaluate DETACH on the independent test set.
7. Compare DETACH with random pruning.
8. Compare ROCKET-based approaches with Ridge classification on raw data.
9. Report Accuracy, F1-score, feature count, prediction time, and confusion matrices.

## Main Output Files

### Result Tables

- comparison_results_4_models.csv
- final_report_table.csv
- summary_results.csv
- report_table_percent.csv
- feature_reduction_results.csv
- detach_vs_random_improvement.csv
- classwise_summary.csv

### Repeated Experiments

- detach_5_runs.csv
- random_pruning_10_runs.csv

### Figures

- accuracy_mean_std_comparison.png
- f1_mean_std_comparison.png
- four_model_accuracy_comparison.png
- four_model_f1_comparison.png
- four_model_feature_count_comparison.png
- detach_confusion_matrix.png
- raw_ridge_confusion_matrix.png
- classwise_f1_comparison.png

### Documentation

- project_summary.txt
- detach_classification_report.txt
- raw_ridge_classification_report.txt
- environment_versions.txt
- requirements.txt

## Reproducibility

Fixed random seeds were used for:

- ROCKET kernel generation
- Random feature selection
- Training-validation splitting

DETACH was evaluated using five ROCKET seeds. Random pruning was evaluated using ten random seeds.

## Conclusion

DETACH reduced the ROCKET feature space by 96.06% while maintaining competitive classification performance. It also outperformed random feature pruning with the same feature budget. The poor performance of Ridge classification on raw FordB signals demonstrates the importance of ROCKET feature extraction.
""").strip()

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_text)

print("README.md saved successfully")

In [ ]:
Z_sub_raw, Z_val_raw, y_sub_clean, y_val_clean = train_test_split(
    Z_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Sub-train shape:", Z_sub_raw.shape)
print("Validation shape:", Z_val_raw.shape)
print("Test shape:", Z_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
Z_sub_raw, Z_val_raw, y_sub_clean, y_val_clean = train_test_split(
    Z_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Sub-train shape:", Z_sub_raw.shape)
print("Validation shape:", Z_val_raw.shape)
print("Test shape:", Z_test.shape)

In [ ]:
clean_scaler = StandardScaler()

Z_sub_clean = clean_scaler.fit_transform(Z_sub_raw)
Z_val_clean = clean_scaler.transform(Z_val_raw)
Z_test_clean = clean_scaler.transform(Z_test)

print("Scaled sub-train:", Z_sub_clean.shape)
print("Scaled validation:", Z_val_clean.shape)
print("Scaled test:", Z_test_clean.shape)

In [ ]:
detach_clean_model = DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)

detach_clean_model.fit(
    Z_sub_clean,
    y_sub_clean,
    Z_val_clean,
    y_val_clean
)

print("Clean DETACH fitted successfully.")

In [ ]:
from detach_rocket.detach_classes import DetachMatrix
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

detach_results = []

for seed in rocket_seeds:
    data = clean_data_by_seed[seed]

    Z_sub = data["subtrain"]
    Z_val = data["validation"]
    Z_test_scaled = data["test"]
    y_sub = data["y_sub"]
    y_val = data["y_val"]

    np.random.seed(42)

    model = DetachMatrix(
        trade_off=0.1,
        recompute_alpha=False,
        verbose=False
    )

    model.fit(
        Z_sub,
        y_sub,
        Z_val,
        y_val
    )

    selected_mask = model._feature_mask

    Z_sub_detach = Z_sub[:, selected_mask]
    Z_test_detach = Z_test_scaled[:, selected_mask]

    ridge = RidgeClassifierCV(
        alphas=np.logspace(-3, 3, 10)
    )

    ridge.fit(Z_sub_detach, y_sub)

    y_pred = ridge.predict(Z_test_detach)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    detach_results.append({
        "seed": seed,
        "features": int(selected_mask.sum()),
        "accuracy": acc,
        "f1": f1,
        "alpha": ridge.alpha_
    })

    print(
        f"seed={seed} | "
        f"features={int(selected_mask.sum())} | "
        f"Accuracy={acc:.4f} | "
        f"F1={f1:.4f}"
    )

detach_df = pd.DataFrame(detach_results)

print()
print(detach_df)

In [ ]:
print(detach_df.to_string(index=False))

print("\nDETACH multi-seed summary")
print(
    "Accuracy:",
    f"{detach_df['accuracy'].mean()*100:.2f}% ± "
    f"{detach_df['accuracy'].std(ddof=1)*100:.2f}%"
)
print(
    "F1-score:",
    f"{detach_df['f1'].mean()*100:.2f}% ± "
    f"{detach_df['f1'].std(ddof=1)*100:.2f}%"
)
print(
    "Selected features:",
    f"{detach_df['features'].mean():.1f} ± "
    f"{detach_df['features'].std(ddof=1):.1f}"
)

In [ ]:
from scipy.stats import ttest_rel

comparison_df = detach_df.merge(
    selectkbest_df,
    on="seed",
    suffixes=("_detach", "_selectkbest")
)

comparison_df["acc_diff"] = (
    comparison_df["accuracy_detach"]
    - comparison_df["accuracy_selectkbest"]
)

comparison_df["f1_diff"] = (
    comparison_df["f1_detach"]
    - comparison_df["f1_selectkbest"]
)

print(
    comparison_df[
        [
            "seed",
            "accuracy_detach",
            "accuracy_selectkbest",
            "acc_diff",
            "f1_detach",
            "f1_selectkbest",
            "f1_diff"
        ]
    ].to_string(index=False)
)

acc_test = ttest_rel(
    comparison_df["accuracy_detach"],
    comparison_df["accuracy_selectkbest"]
)

f1_test = ttest_rel(
    comparison_df["f1_detach"],
    comparison_df["f1_selectkbest"]
)

print("\nMean Accuracy difference:",
      f"{comparison_df['acc_diff'].mean()*100:.2f} percentage points")

print("Mean F1 difference:",
      f"{comparison_df['f1_diff'].mean()*100:.2f} percentage points")

print("\nPaired t-test Accuracy:")
print(acc_test)

print("\nPaired t-test F1:")
print(f1_test)

In [ ]:
matched_selectkbest_results = []

for seed in rocket_seeds:
    data = clean_data_by_seed[seed]

    Z_sub = data["subtrain"]
    Z_test_scaled = data["test"]
    y_sub = data["y_sub"]

    k = int(
        detach_df.loc[
            detach_df["seed"] == seed,
            "features"
        ].iloc[0]
    )

    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    Z_sub_sel = selector.fit_transform(Z_sub, y_sub)
    Z_test_sel = selector.transform(Z_test_scaled)

    model = RidgeClassifierCV(
        alphas=np.logspace(-3, 3, 10)
    )

    model.fit(Z_sub_sel, y_sub)
    y_pred = model.predict(Z_test_sel)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    matched_selectkbest_results.append({
        "seed": seed,
        "features": k,
        "accuracy": acc,
        "f1": f1,
        "alpha": model.alpha_
    })

    print(
        f"seed={seed} | "
        f"features={k} | "
        f"Accuracy={acc:.4f} | "
        f"F1={f1:.4f}"
    )

matched_selectkbest_df = pd.DataFrame(
    matched_selectkbest_results
)

print()
print(matched_selectkbest_df.to_string(index=False))

In [ ]:
matched_comparison_df = detach_df.merge(
    matched_selectkbest_df,
    on=["seed", "features"],
    suffixes=("_detach", "_selectkbest")
)

matched_comparison_df["acc_diff"] = (
    matched_comparison_df["accuracy_detach"]
    - matched_comparison_df["accuracy_selectkbest"]
)

matched_comparison_df["f1_diff"] = (
    matched_comparison_df["f1_detach"]
    - matched_comparison_df["f1_selectkbest"]
)

print(
    matched_comparison_df[
        [
            "seed",
            "features",
            "accuracy_detach",
            "accuracy_selectkbest",
            "acc_diff",
            "f1_detach",
            "f1_selectkbest",
            "f1_diff"
        ]
    ].to_string(index=False)
)

acc_test_matched = ttest_rel(
    matched_comparison_df["accuracy_detach"],
    matched_comparison_df["accuracy_selectkbest"]
)

f1_test_matched = ttest_rel(
    matched_comparison_df["f1_detach"],
    matched_comparison_df["f1_selectkbest"]
)

print("\nMean Accuracy difference:",
      f"{matched_comparison_df['acc_diff'].mean()*100:.2f} percentage points")

print("Mean F1 difference:",
      f"{matched_comparison_df['f1_diff'].mean()*100:.2f} percentage points")

print("\nPaired t-test Accuracy:")
print(acc_test_matched)

print("\nPaired t-test F1:")
print(f1_test_matched)

In [ ]:
matched_comparison_df.to_csv(
    "matched_budget_detach_vs_selectkbest_5seeds.csv",
    index=False
)

print("Saved matched-budget results.")

In [ ]:
detach_df.to_csv(
    "detach_multiseed_5seeds.csv",
    index=False
)

matched_selectkbest_df.to_csv(
    "selectkbest_matched_budget_5seeds.csv",
    index=False
)

print("Saved DETACH and matched SelectKBest results.")

In [ ]:
summary_stats = pd.DataFrame({
    "method": ["DETACH", "SelectKBest matched-budget"],
    "accuracy_mean": [
        detach_df["accuracy"].mean(),
        matched_selectkbest_df["accuracy"].mean()
    ],
    "accuracy_std": [
        detach_df["accuracy"].std(ddof=1),
        matched_selectkbest_df["accuracy"].std(ddof=1)
    ],
    "f1_mean": [
        detach_df["f1"].mean(),
        matched_selectkbest_df["f1"].mean()
    ],
    "f1_std": [
        detach_df["f1"].std(ddof=1),
        matched_selectkbest_df["f1"].std(ddof=1)
    ]
})

summary_stats.to_csv(
    "multiseed_matched_budget_summary.csv",
    index=False
)

print(summary_stats.to_string(index=False))

In [ ]:
detach_clean_pred = detach_clean_model.predict(
    Z_test_clean
)

detach_clean_accuracy = accuracy_score(
    y_test,
    detach_clean_pred
)

detach_clean_f1 = f1_score(
    y_test,
    detach_clean_pred
)

print("Clean DETACH Accuracy:", detach_clean_accuracy)
print("Clean DETACH F1-score:", detach_clean_f1)

In [ ]:
clean_selected_features = detach_clean_model.support_.sum()

print("Selected features:", clean_selected_features)
print(
    "Feature reduction:",
    round(
        (1 - clean_selected_features / Z_sub_clean.shape[1]) * 100,
        2
    ),
    "%"
)

In [ ]:
print([
    name for name in dir(detach_clean_model)
    if not name.startswith("_")
])

In [ ]:
print(detach_clean_model.__dict__)

In [ ]:
clean_selected_features = detach_clean_model._feature_mask.sum()

print("Selected features:", clean_selected_features)
print(
    "Feature reduction:",
    round(
        (1 - clean_selected_features / Z_sub_clean.shape[1]) * 100,
        2
    ),
    "%"
)

In [ ]:
clean_detach_results = {
    "model": "Clean DETACH-ROCKET",
    "num_features": int(clean_selected_features),
    "accuracy": detach_clean_accuracy,
    "f1_score": detach_clean_f1,
    "feature_reduction_percent": (
        1 - clean_selected_features / Z_sub_clean.shape[1]
    ) * 100
}

clean_detach_results

In [ ]:
clean_detach_df = pd.DataFrame([clean_detach_results])

clean_detach_df.to_csv(
    "clean_detach_results.csv",
    index=False
)

print("Clean DETACH results saved successfully")

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
select_k_best = SelectKBest(
    score_func=f_classif,
    k=clean_selected_features
)

Z_sub_kbest = select_k_best.fit_transform(
    Z_sub_clean,
    y_sub_clean
)

Z_val_kbest = select_k_best.transform(
    Z_val_clean
)

Z_test_kbest = select_k_best.transform(
    Z_test_clean
)

print("KBest sub-train:", Z_sub_kbest.shape)
print("KBest validation:", Z_val_kbest.shape)
print("KBest test:", Z_test_kbest.shape)

In [ ]:
kbest_model = RidgeClassifierCV(alphas=alphas)

kbest_model.fit(
    Z_sub_kbest,
    y_sub_clean
)

kbest_pred = kbest_model.predict(
    Z_test_kbest
)

kbest_accuracy = accuracy_score(
    y_test,
    kbest_pred
)

kbest_f1 = f1_score(
    y_test,
    kbest_pred
)

print("SelectKBest Accuracy:", kbest_accuracy)
print("SelectKBest F1-score:", kbest_f1)
print("Selected Alpha:", kbest_model.alpha_)

In [ ]:
kbest_results = {
    "model": "SelectKBest",
    "num_features": Z_sub_kbest.shape[1],
    "accuracy": kbest_accuracy,
    "f1_score": kbest_f1,
    "alpha": kbest_model.alpha_
}

kbest_results

In [ ]:
kbest_df = pd.DataFrame([kbest_results])

kbest_df.to_csv(
    "select_kbest_results.csv",
    index=False
)

print("SelectKBest results saved successfully")

In [ ]:
clean_feature_selection_comparison = pd.DataFrame([
    {
        "Method": "DETACH-ROCKET",
        "Features": clean_detach_results["num_features"],
        "Accuracy": clean_detach_results["accuracy"],
        "F1-score": clean_detach_results["f1_score"]
    },
    {
        "Method": "SelectKBest",
        "Features": kbest_results["num_features"],
        "Accuracy": kbest_results["accuracy"],
        "F1-score": kbest_results["f1_score"]
    }
])

clean_feature_selection_comparison

In [ ]:
clean_feature_selection_comparison.to_csv(
    "clean_feature_selection_comparison.csv",
    index=False
)

print("Clean feature-selection comparison saved successfully")

In [ ]:
kbest_accuracy_gap = (
    clean_detach_results["accuracy"]
    - kbest_results["accuracy"]
) * 100

kbest_f1_gap = (
    clean_detach_results["f1_score"]
    - kbest_results["f1_score"]
) * 100

print(
    "DETACH Accuracy advantage:",
    round(kbest_accuracy_gap, 2),
    "percentage points"
)

print(
    "DETACH F1-score advantage:",
    round(kbest_f1_gap, 2),
    "percentage points"
)

In [ ]:
detach_vs_kbest = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "DETACH_advantage_percentage_points": kbest_accuracy_gap
    },
    {
        "Metric": "F1-score",
        "DETACH_advantage_percentage_points": kbest_f1_gap
    }
])

detach_vs_kbest.to_csv(
    "detach_vs_selectkbest_improvement.csv",
    index=False
)

print("DETACH vs SelectKBest improvement saved successfully")

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    clean_feature_selection_comparison["Method"],
    clean_feature_selection_comparison["Accuracy"]
)

plt.ylabel("Accuracy")
plt.title("DETACH-ROCKET vs SelectKBest")
plt.ylim(0.78, 0.83)
plt.tight_layout()

plt.savefig(
    "detach_vs_selectkbest_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    clean_feature_selection_comparison["Method"],
    clean_feature_selection_comparison["F1-score"]
)

plt.ylabel("F1-score")
plt.title("DETACH-ROCKET vs SelectKBest")
plt.ylim(0.78, 0.83)
plt.tight_layout()

plt.savefig(
    "detach_vs_selectkbest_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
updated_comparison_df = pd.DataFrame([
    {
        "Model": "Full ROCKET",
        "Features": baseline_results["num_features"],
        "Accuracy": baseline_results["accuracy"],
        "F1-score": baseline_results["f1_score"]
    },
    {
        "Model": "DETACH-ROCKET",
        "Features": clean_detach_results["num_features"],
        "Accuracy": clean_detach_results["accuracy"],
        "F1-score": clean_detach_results["f1_score"]
    },
    {
        "Model": "Random Pruning",
        "Features": selected_features,
        "Accuracy": random_runs_df["accuracy"].mean(),
        "F1-score": random_runs_df["f1_score"].mean()
    },
    {
        "Model": "SelectKBest",
        "Features": kbest_results["num_features"],
        "Accuracy": kbest_results["accuracy"],
        "F1-score": kbest_results["f1_score"]
    },
    {
        "Model": "Raw Ridge",
        "Features": raw_ridge_results["num_features"],
        "Accuracy": raw_ridge_results["accuracy"],
        "F1-score": raw_ridge_results["f1_score"]
    }
])

updated_comparison_df

In [ ]:
updated_comparison_df.to_csv(
    "updated_comparison_5_models.csv",
    index=False
)

print("Updated five-model comparison saved successfully")

In [ ]:
updated_report_table = updated_comparison_df.copy()

updated_report_table["Accuracy (%)"] = (
    updated_report_table["Accuracy"] * 100
).round(2)

updated_report_table["F1-score (%)"] = (
    updated_report_table["F1-score"] * 100
).round(2)

updated_report_table = updated_report_table[
    [
        "Model",
        "Features",
        "Accuracy (%)",
        "F1-score (%)"
    ]
]

updated_report_table

In [ ]:
updated_report_table.to_csv(
    "updated_report_table_percent.csv",
    index=False
)

print("Updated report table saved successfully")

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    updated_comparison_df["Model"],
    updated_comparison_df["Accuracy"]
)

plt.ylabel("Accuracy")
plt.title("Accuracy Comparison of Five Methods")
plt.ylim(0.45, 0.85)
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    "updated_five_model_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    updated_comparison_df["Model"],
    updated_comparison_df["F1-score"]
)

plt.ylabel("F1-score")
plt.title("F1-score Comparison of Five Methods")
plt.ylim(0.45, 0.85)
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    "updated_five_model_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    updated_comparison_df["Model"],
    updated_comparison_df["Features"]
)

plt.ylabel("Number of Features")
plt.title("Feature Count Comparison of Five Methods")
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    "updated_five_model_feature_count.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
updated_project_summary = f"""
Updated FordB Project Summary

Full ROCKET:
- Features: {baseline_results['num_features']}
- Accuracy: {baseline_results['accuracy'] * 100:.2f}%
- F1-score: {baseline_results['f1_score'] * 100:.2f}%

Clean DETACH-ROCKET:
- Features: {clean_detach_results['num_features']}
- Accuracy: {clean_detach_results['accuracy'] * 100:.2f}%
- F1-score: {clean_detach_results['f1_score'] * 100:.2f}%
- Feature reduction: {clean_detach_results['feature_reduction_percent']:.2f}%

Random Pruning:
- Features: {selected_features}
- Mean Accuracy: {random_runs_df['accuracy'].mean() * 100:.2f}%
- Mean F1-score: {random_runs_df['f1_score'].mean() * 100:.2f}%

SelectKBest:
- Features: {kbest_results['num_features']}
- Accuracy: {kbest_results['accuracy'] * 100:.2f}%
- F1-score: {kbest_results['f1_score'] * 100:.2f}%

Raw Ridge:
- Features: {raw_ridge_results['num_features']}
- Accuracy: {raw_ridge_results['accuracy'] * 100:.2f}%
- F1-score: {raw_ridge_results['f1_score'] * 100:.2f}%

DETACH advantage over SelectKBest:
- Accuracy: {kbest_accuracy_gap:.2f} percentage points
- F1-score: {kbest_f1_gap:.2f} percentage points
"""

with open(
    "updated_project_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(updated_project_summary)

print(updated_project_summary)
print("Updated project summary saved successfully")

In [ ]:
clean_random_seeds = list(range(10))
clean_random_runs = []

print("Number of clean random runs:", len(clean_random_seeds))
print("Feature budget:", clean_selected_features)

In [ ]:
for seed in clean_random_seeds:
    rng = np.random.default_rng(seed)

    random_indices = rng.choice(
        Z_sub_clean.shape[1],
        size=clean_selected_features,
        replace=False
    )

    model = RidgeClassifierCV(alphas=alphas)

    model.fit(
        Z_sub_clean[:, random_indices],
        y_sub_clean
    )

    predictions = model.predict(
        Z_test_clean[:, random_indices]
    )

    clean_random_runs.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, predictions),
        "f1_score": f1_score(y_test, predictions),
        "alpha": model.alpha_
    })

print("Completed clean random runs:", len(clean_random_runs))

In [ ]:
clean_random_runs_df = pd.DataFrame(clean_random_runs)

print(clean_random_runs_df)

print("\nMean Accuracy:", clean_random_runs_df["accuracy"].mean())
print("Std Accuracy:", clean_random_runs_df["accuracy"].std())

print("\nMean F1-score:", clean_random_runs_df["f1_score"].mean())
print("Std F1-score:", clean_random_runs_df["f1_score"].std())

In [ ]:
clean_random_runs_df.to_csv(
    "clean_random_pruning_10_runs.csv",
    index=False
)

print("Clean random pruning results saved successfully")

In [ ]:
final_clean_comparison = pd.DataFrame([
    {
        "Method": "DETACH-ROCKET",
        "Features": clean_detach_results["num_features"],
        "Accuracy": clean_detach_results["accuracy"],
        "F1-score": clean_detach_results["f1_score"]
    },
    {
        "Method": "SelectKBest",
        "Features": kbest_results["num_features"],
        "Accuracy": kbest_results["accuracy"],
        "F1-score": kbest_results["f1_score"]
    },
    {
        "Method": "Random Pruning",
        "Features": clean_selected_features,
        "Accuracy": clean_random_runs_df["accuracy"].mean(),
        "F1-score": clean_random_runs_df["f1_score"].mean()
    }
])

final_clean_comparison

In [ ]:
final_clean_report_table = final_clean_comparison.copy()

final_clean_report_table["Accuracy (%)"] = (
    final_clean_report_table["Accuracy"] * 100
).round(2)

final_clean_report_table["F1-score (%)"] = (
    final_clean_report_table["F1-score"] * 100
).round(2)

final_clean_report_table = final_clean_report_table[
    ["Method", "Features", "Accuracy (%)", "F1-score (%)"]
]

final_clean_report_table.to_csv(
    "final_clean_feature_selection_comparison.csv",
    index=False
)

final_clean_report_table

In [ ]:
final_clean_improvements = pd.DataFrame([
    {
        "Comparison": "DETACH vs SelectKBest",
        "Accuracy improvement (percentage points)": (
            clean_detach_results["accuracy"] - kbest_results["accuracy"]
        ) * 100,
        "F1 improvement (percentage points)": (
            clean_detach_results["f1_score"] - kbest_results["f1_score"]
        ) * 100
    },
    {
        "Comparison": "DETACH vs Random Pruning",
        "Accuracy improvement (percentage points)": (
            clean_detach_results["accuracy"]
            - clean_random_runs_df["accuracy"].mean()
        ) * 100,
        "F1 improvement (percentage points)": (
            clean_detach_results["f1_score"]
            - clean_random_runs_df["f1_score"].mean()
        ) * 100
    }
])

final_clean_improvements = final_clean_improvements.round(2)

final_clean_improvements.to_csv(
    "final_clean_improvements.csv",
    index=False
)

final_clean_improvements

In [ ]:
plot_data = final_clean_report_table.set_index("Method")[
    ["Accuracy (%)", "F1-score (%)"]
]

ax = plot_data.plot(
    kind="bar",
    figsize=(9, 5),
    width=0.75
)

ax.set_title("Clean Feature-Selection Comparison on FordB")
ax.set_ylabel("Performance (%)")
ax.set_xlabel("")
ax.set_ylim(75, 83)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Metric")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=3,
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "final_clean_feature_selection_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd

selected_features_count = 789
initial_features_count = 20000

compression_analysis = {
    "initial_features": initial_features_count,
    "selected_features": selected_features_count,
    "removed_features": (
        initial_features_count - selected_features_count
    ),
    "retained_features_percent": (
        selected_features_count / initial_features_count
    ) * 100,
    "feature_reduction_percent": (
        1 - selected_features_count / initial_features_count
    ) * 100,
    "compression_ratio": (
        initial_features_count / selected_features_count
    )
}

compression_analysis_df = pd.DataFrame(
    [compression_analysis]
).round(3)

compression_analysis_df.to_csv(
    "detach_compression_analysis.csv",
    index=False
)

compression_analysis_df

In [ ]:
from pathlib import Path

readme_text = r'''# DETACH-ROCKET on the FordB Dataset

## Project Overview

This project reproduces and extends the DETACH-ROCKET feature-selection
pipeline for binary time-series classification on the FordB dataset.

ROCKET transforms each time series into a high-dimensional representation
using random convolutional kernels. DETACH then performs sequential,
coefficient-based feature pruning to identify a compact subset of informative
ROCKET features.

The project evaluates five approaches:

1. Full ROCKET using all 20,000 transformed features
2. DETACH-ROCKET feature selection
3. SelectKBest with the same feature budget as DETACH
4. Random feature pruning with the same feature budget as DETACH
5. RidgeClassifierCV applied directly to the raw time-series samples

The main controlled feature-selection comparison uses the same sub-training
partition, validation partition, fitted scaler, test set, and feature budget
for DETACH, SelectKBest, and Random Pruning.

---

## Dataset

FordB is a binary time-series classification dataset from the UCR archive.

- Training samples: 3,636
- Test samples: 810
- Time points per sample: 500
- Classes: -1 and +1
- Missing values: none

Class distribution:

| Partition | Class -1 | Class +1 |
|---|---:|---:|
| Train | 1,860 | 1,776 |
| Test | 401 | 409 |

The two classes are approximately balanced.

---

## ROCKET Transformation

ROCKET applies 10,000 random convolutional kernels to each time series.

Two features are extracted from each kernel response:

- MAX: maximum convolution response
- PPV: proportion of positive values

This produces 20,000 features for each sample.

Transformation dimensions:

- Training set: 3,636 × 500 → 3,636 × 20,000
- Test set: 810 × 500 → 810 × 20,000

---

## Leakage-Free Experimental Pipeline

The final experiment uses a leakage-free preprocessing pipeline:

1. Load the original FordB train and test partitions.
2. Generate 20,000 ROCKET features.
3. Split the ROCKET training features into stratified sub-training and validation partitions.
4. Fit `StandardScaler` only on the sub-training partition.
5. Transform the validation and test partitions using the fitted scaler.
6. Run DETACH using the sub-training and validation partitions.
7. Fit SelectKBest only on the same sub-training partition.
8. Run Random Pruning using the same 789-feature budget.
9. Evaluate the final models on the independent test set.
10. Use the test set only for final evaluation.

Final split dimensions:

- Sub-training: 2,908 × 20,000
- Validation: 728 × 20,000
- Test: 810 × 20,000

After correcting the scaling order, the DETACH result remained unchanged.

---

## DETACH Configuration

The final DETACH model was configured as follows:

```python
DetachMatrix(
    trade_off=0.1,
    recompute_alpha=False,
    verbose=True
)
```

DETACH sequentially removes low-importance features based on the absolute
coefficients of a Ridge classifier. Validation performance is used to select
the final model size.

---

## Final Results

### Overall Model Summary

| Method | Features | Accuracy | F1-score |
|---|---:|---:|---:|
| Full ROCKET | 20,000 | 80.25% | 80.58% |
| DETACH-ROCKET | 789 | **81.48%** | **81.53%** |
| SelectKBest | 789 | 80.62% | 80.92% |
| Random Pruning | 789 | 78.91% ± 1.01% | 78.85% ± 0.95% |
| Raw Ridge | 500 | 48.89% | 47.06% |

Random Pruning results are reported as mean ± standard deviation over
10 independent random seeds.

---

## Controlled Feature-Selection Comparison

DETACH, SelectKBest, and Random Pruning were compared under the same
experimental conditions and with exactly 789 selected features.

| Method | Features | Accuracy | F1-score |
|---|---:|---:|---:|
| DETACH-ROCKET | 789 | **81.48%** | **81.53%** |
| SelectKBest | 789 | 80.62% | 80.92% |
| Random Pruning | 789 | 78.91% | 78.85% |

DETACH improvement over SelectKBest:

- Accuracy: +0.86 percentage points
- F1-score: +0.60 percentage points

DETACH improvement over Random Pruning:

- Accuracy: +2.57 percentage points
- F1-score: +2.68 percentage points

Because the three methods used the same feature budget and preprocessing
pipeline, the results provide a controlled comparison of the selected
feature subsets.

---

## Feature Compression

DETACH reduced the ROCKET feature space from 20,000 to 789 features.

- Initial features: 20,000
- Selected features: 789
- Removed features: 19,211
- Retained features: 3.945%
- Feature reduction: 96.055%
- Compression ratio: 25.35×

The final representation was approximately 25.35 times smaller than the
original ROCKET representation.

Test accuracy increased numerically from 80.25% for Full ROCKET to 81.48%
for DETACH-ROCKET.

---

## Interpretation

The results support four main observations:

1. ROCKET feature extraction is essential for FordB. Ridge classification
   on the raw 500-point signals achieved only 48.89% accuracy.

2. The 20,000-dimensional ROCKET representation contains substantial
   redundancy for this dataset.

3. Randomly retaining 789 features did not reproduce DETACH performance.

4. Under the controlled 789-feature comparison, DETACH achieved higher
   Accuracy and F1-score than both SelectKBest and Random Pruning.

---

## Extensions Added in This Project

Beyond reproducing the core DETACH-ROCKET pipeline, this project adds:

- F1-score evaluation
- Ridge classification on the raw time-series values
- SelectKBest with the same 789-feature budget
- Clean Random Pruning over 10 independent seeds
- Leakage-free scaling
- A controlled equal-budget feature-selection comparison
- Feature-compression analysis
- Reproducible CSV result tables
- Updated performance and feature-count figures
- Explicit separation between the original paper and the project extensions

---

## Installation

Create and activate a Python virtual environment, then install the required
packages:

```bash
pip install -r requirements.txt
```

The experiments were developed using Python 3.10.

---

## Notebook

The main experimental notebook is:

```text
examples/Detach_ROCKET_example_UCR.ipynb
```

---

## Main Output Files

### Final Result Tables

- `clean_detach_results.csv`
- `select_kbest_results.csv`
- `clean_random_pruning_10_runs.csv`
- `final_clean_feature_selection_comparison.csv`
- `final_clean_improvements.csv`
- `updated_comparison_5_models.csv`
- `updated_report_table_percent.csv`
- `detach_compression_analysis.csv`

### Final Figures

- `final_clean_feature_selection_comparison.png`
- `updated_five_model_accuracy.png`
- `updated_five_model_f1.png`
- `updated_five_model_feature_count.png`
- `detach_vs_selectkbest_accuracy.png`
- `detach_vs_selectkbest_f1.png`

### Documentation

- `updated_project_summary.txt`
- `environment_versions.txt`
- `requirements.txt`

Some earlier exploratory files may remain in the project directory, but the
files listed above contain the final leakage-free results used in the report
and presentation.

---

## Reproducibility

Fixed random seeds were used for:

- Training-validation splitting
- Random feature selection
- ROCKET kernel generation in repeated exploratory experiments

The final clean split used stratification and `random_state=42`.

Random Pruning was evaluated using 10 independent seeds:

```text
0, 1, 2, 3, 4, 5, 6, 7, 8, 9
```

The test set was not used for fitting the scaler, selecting features, or
choosing the final DETACH model size.

---

## Conclusion

DETACH-ROCKET reduced the FordB ROCKET feature space from 20,000 to 789
features, corresponding to a 96.055% reduction and a 25.35-fold compression.

The final DETACH model achieved:

- 81.48% Accuracy
- 81.53% F1-score

In the controlled comparison with the same 789-feature budget, DETACH
outperformed both SelectKBest and Random Pruning on the FordB test set.

These results show that targeted feature selection can produce a substantially
more compact ROCKET representation without sacrificing classification
performance on FordB.
'''

readme_path = Path("README.md")

readme_path.write_text(
    readme_text,
    encoding="utf-8"
)

print("README replaced successfully.")
print("Saved to:", readme_path.resolve())

In [20]:
print(summary_stats.to_string(index=False))

print("\nMean Accuracy difference:",
      f"{matched_comparison_df['acc_diff'].mean()*100:.2f} percentage points")

print("Mean F1 difference:",
      f"{matched_comparison_df['f1_diff'].mean()*100:.2f} percentage points")

print("\nPaired t-test Accuracy:")
print(acc_test_matched)

print("\nPaired t-test F1:")
print(f1_test_matched)

                    method  accuracy_mean  accuracy_std  f1_mean   f1_std
                    DETACH       0.809136      0.010534 0.809673 0.009990
SelectKBest matched-budget       0.775556      0.014618 0.786624 0.008557

Mean Accuracy difference: 3.36 percentage points
Mean F1 difference: 2.30 percentage points

Paired t-test Accuracy:
TtestResult(statistic=4.084802466005891, pvalue=0.015039060496915991, df=4)

Paired t-test F1:
TtestResult(statistic=3.609497005321674, pvalue=0.022566692922403504, df=4)


## Prepare Dataset Matrices

In [ ]:
import numpy as np

# Create data matrices and remove possible rows with nans

print(f"Dataset Matrix Shape: ( # of instances , time series length )")
print(f" ")

# Train Matrix
X_train = current_dataset['data_train']
print(f"Train: {X_train.shape}")
non_nan_mask_train = ~np.isnan(X_train).any(axis=1)
non_inf_mask_train = ~np.isinf(X_train).any(axis=1)
mask_train = np.logical_and(non_nan_mask_train,non_inf_mask_train)
X_train = X_train[mask_train]
X_train = X_train.reshape(X_train.shape[0],1,X_train.shape[1])
y_train = current_dataset['target_train']
y_train = y_train[mask_train]

print(f" ")

# Test Matrix
X_test = current_dataset['data_test']
#print(f"Number of test instances: {len(X_test)}")
print(f"Test: {X_test.shape}")
non_nan_mask_test = ~np.isnan(X_test).any(axis=1)
non_inf_mask_test = ~np.isinf(X_test).any(axis=1)
mask_test = np.logical_and(non_nan_mask_test,non_inf_mask_test)
X_test = X_test[mask_test]
X_test = X_test.reshape(X_test.shape[0],1,X_test.shape[1])
y_test = current_dataset['target_test']
y_test = y_test[mask_test]

## Train and Evaluate the Model

In [ ]:
from detach_rocket.detach_classes import DetachRocket

np.random.seed(2)

# Select initial model characteristics
model_type = "rocket"
num_kernels = 10000

# Create model object
DetachRocketModel = DetachRocket(model_type, num_kernels=num_kernels)

# Trian Model
DetachRocketModel.fit(X_train,y_train)

# Evaluate Performance on Test Set
detach_test_score, full_test_score= DetachRocketModel.score(X_test,y_test)
print('Test Accuraccy Full Model: {:.2f}%'.format(100*full_test_score))
print('Test Accuraccy Detach-ROCKET: {:.2f}%'.format(100*detach_test_score))

## Plot SFD Curve and Optimal Model Selection

In [ ]:
import matplotlib.pyplot as plt

percentage_vector = DetachRocketModel._percentage_vector
acc_curve = DetachRocketModel._sfd_curve

c = DetachRocketModel.trade_off

x=(percentage_vector) * 100
y=(acc_curve/acc_curve[0]-1) * 100

point_x = x[DetachRocketModel._max_index]
#point_y = y[DetachRocketModel._max_index]

plt.figure(figsize=(8,3.5))
plt.axvline(x = point_x, color = 'r',label=f'Optimal Model (c={c})')
plt.plot(x, y, label='SFD curve', linewidth=2.5, color='C7', alpha=1)
#plt.scatter(point_x, point_y, s=50, marker='o', label=f'Optimal point (c={c})')

plt.grid(True, linestyle='-', alpha=0.5)
plt.xlim(102,-2)
plt.xlabel('% of Retained Features')
plt.ylabel('Relative Validation Set Accuracy (%)')
plt.legend()
plt.show()

print('Optimal Model Size: {:.2f}% of full model'.format(point_x))